In [ ]:
from hardware import SLMManager
# --- Global hardware manager ---

# Simulation mode (without connecting SLM)
slm_manager = SLMManager(sim_mode=True)

# # Normal mode (with SLM)
# slm_manager = SLMManager(sim_mode=False) 

#

🎬 Simulation Mode Enabled


In [3]:
import numpy as np
import ipywidgets as widgets
from IPython.display import display, clear_output
import torch

# Project modules
import config
from optics_utils import create_gaussian_template, process_parameters
from phase_generators import generate_fresnel_pattern, generate_optimized_pattern
from visualization import plot_live_update, plot_final_results, plot_fresnel_pattern

# Make sure matplotlib works in notebooks (safe if not in IPython)
try:
    get_ipython().run_line_magic("matplotlib", "inline")
except Exception:
    pass


# --- UI builder ---
def build_ui():
    # --- Mode selector ---
    mode_selector = widgets.ToggleButtons(
        options=['Optimized Microlens', 'Fresnel Microlens'],
        description='Mode:', 
        style={'description_width': 'initial'},
        layout=widgets.Layout(width='400px')
    )

    # --- Auto-update checkbox ---
    auto_update_checkbox = widgets.Checkbox(
        value=False,
        description='Auto Update (Live)',
        style={'description_width': 'initial'},
        layout=widgets.Layout(width='200px')
    )

    # --- Common controls for both modes ---
    common_widgets = {
        'focal_length_coarse': widgets.FloatText(
            value=config.COMMON_DEFAULTS['focal_length_coarse'],
            description='Focal Length (mm):',
            style={'description_width': '130px'},
            layout=widgets.Layout(width='240px')
        ),
        'focal_length_fine': widgets.FloatSlider(
            value=config.COMMON_DEFAULTS['focal_length_fine'],
            min=-5.0,
            max=5.0,
            step=0.1,
            description='Fine Adjust (mm):',
            style={'description_width': '130px'},
            layout=widgets.Layout(width='340px')
        ),
        'M': widgets.IntText(
            value=config.COMMON_DEFAULTS['M'],
            description='Array Scale M*M:',
            style={'description_width': '80px'},
            layout=widgets.Layout(width='200px')
        ),
        'two_pi_value': widgets.IntText(
            value=config.COMMON_DEFAULTS['two_pi_value'],
            description='Gray Value for 2π:',
            style={'description_width': '130px'},
            layout=widgets.Layout(width='240px')
        ),
    }
    
    # --- ROI controls ---
    # Build options from config
    roi_options = [(f'{k} ({v}×{v})', v) for k, v in config.ROI_SIZE_OPTIONS.items()]
    
    roi_selector = widgets.Dropdown(
        options=roi_options,
        value=config.ROI_DEFAULT_SIZE,
        description='Quick Set:',
        style={'description_width': '70px'},
        layout=widgets.Layout(width='250px')
    )
    
    roi_custom = widgets.IntText(
        value=config.ROI_DEFAULT_SIZE,
        description='ROI Size (square):',
        style={'description_width': '130px'},
        layout=widgets.Layout(width='240px')
    )
    
    # --- ROI Position Controls ---
    # Get SLM shape for setting valid ranges
    slm_height, slm_width = slm_manager.shape
    
    # Initialize at center
    roi_center_x = widgets.IntSlider(
        value=slm_width // 2,
        min=0,
        max=slm_width,
        step=1,
        description='X:',
        style={'description_width': '20px'},
        layout=widgets.Layout(width='160px')
    )
    
    roi_center_y = widgets.IntSlider(
        value=slm_height // 2,
        min=0,
        max=slm_height,
        step=1,
        description='Y:',
        style={'description_width': '20px'},
        layout=widgets.Layout(width='160px')
    )
    
    # Center button
    def center_roi(_b):
        roi_center_x.value = slm_width // 2
        roi_center_y.value = slm_height // 2
    
    center_roi_button = widgets.Button(
        description='⊕',
        button_style='info',
        tooltip='Center ROI',
        layout=widgets.Layout(width='35px')
    )
    center_roi_button.on_click(center_roi)
    
    # Function to update slider ranges based on ROI size
    def update_roi_position_limits():
        half_roi = roi_custom.value // 2
        # Update X slider limits
        roi_center_x.min = half_roi
        roi_center_x.max = slm_width - half_roi
        # Clamp current value if needed
        if roi_center_x.value < roi_center_x.min:
            roi_center_x.value = roi_center_x.min
        elif roi_center_x.value > roi_center_x.max:
            roi_center_x.value = roi_center_x.max
        
        # Update Y slider limits
        roi_center_y.min = half_roi
        roi_center_y.max = slm_height - half_roi
        # Clamp current value if needed
        if roi_center_y.value < roi_center_y.min:
            roi_center_y.value = roi_center_y.min
        elif roi_center_y.value > roi_center_y.max:
            roi_center_y.value = roi_center_y.max
    
    def on_roi_dropdown_change(change):
        roi_custom.value = change.new
        update_roi_position_limits()
    
    def on_roi_custom_change(change):
        update_roi_position_limits()
    
    roi_selector.observe(on_roi_dropdown_change, names='value')
    roi_custom.observe(on_roi_custom_change, names='value')
    
    # --- Optimized mode specific controls ---
    opt_specific_widgets = {
        'overlap_ratio': widgets.FloatText(
            value=config.OPTIMIZED_DEFAULTS['overlap_ratio'],
            description='Overlap Ratio:',
            step=0.1,
            style={'description_width': '110px'},
            layout=widgets.Layout(width='160px')
        ),
        'center_blend': widgets.FloatText(
            value=config.OPTIMIZED_DEFAULTS['center_blend'],
            description='Center blend (0:Orig; 1:Aligned w/μLens):',
            step=0.1,
            style={'description_width': '250px'},
            layout=widgets.Layout(width='300px')
        ),
        'interleaving': widgets.Dropdown(
            options=['coarse1','coarse2','coarse3','checkerboard'],
            value=config.OPTIMIZED_DEFAULTS['interleaving'],
            description='Mask Interleaving Method:',
            style={'description_width': '160px'},
            layout=widgets.Layout(width='250px')
        ),
        'mask_count': widgets.FloatText(
            value=config.OPTIMIZED_DEFAULTS['mask_count'],
            description='Number of Masks:',
            style={'description_width': '110px'},
            layout=widgets.Layout(width='160px')
        ),
        'airy_correction': widgets.FloatText(
            value=config.OPTIMIZED_DEFAULTS['airy_correction'],
            description='Airy Correction:',
            step=0.05,
            style={'description_width': '100px'},
            layout=widgets.Layout(width='160px')
        ),
        'dof_correction': widgets.FloatText(
            value=config.OPTIMIZED_DEFAULTS['dof_correction'],
            description='DOF Correction:',
            step=0.05,
            style={'description_width': '100px'},
            layout=widgets.Layout(width='160px')
        ),
        'psf_energy_level': widgets.FloatText(
            value=config.OPTIMIZED_DEFAULTS['psf_energy_level'],
            step=0.01,
            description='PSF Energy:',
            style={'description_width': '80px'},
            layout=widgets.Layout(width='160px')
        ),
        # 'lr': widgets.FloatText(
        #     value=config.OPTIMIZED_DEFAULTS['lr'],
        #     step=0.001,
        #     description='Learning Rate:',
        #     style={'description_width': '100px'},
        #     layout=widgets.Layout(width='180px')
        # ),
        'lr': widgets.FloatLogSlider(
            value=config.OPTIMIZED_DEFAULTS['lr'],
            base=10,
            min=-5, # min exponent of base
            max=1, # max exponent of base
            step=0.01, # exponent step
            description='Learning Rate:',
            style={'description_width': '100px'},
            layout=widgets.Layout(width='300px')
        ),
        'ni': widgets.IntText(
            value=config.OPTIMIZED_DEFAULTS['ni'],
            description='Iterations:',
            step=500,
            style={'description_width': '100px'},
            layout=widgets.Layout(width='180px')
        )
    }
    
    def show_mask(_b):
        with output_widget:
            clear_output(wait=True)
            # obtain every parameter from fields in widgets
            mode = mode_selector.value
            params = update_parameters(mode)
            # visulize template by calling the function
            create_gaussian_template(
                params['N'], params['N']*config.PIXEL_SIZE, 
                params['focal_length'], 
                config.WAVELENGTH, 
                params['M'], 
                params['overlap_ratio'], torch.device("cpu"),
                airy_correction = params['airy_correction'], 
                center_blend = params['center_blend'], 
                mask_count=int(params['mask_count']), 
                visualize = True,
                interleaving=params['interleaving']
            )
    
    show_mask_button = widgets.Button(
            description='Show masks and tiles',
            button_style='info',
            tooltip='Center ROI',
            layout=widgets.Layout(width='170px')
    )
    show_mask_button.on_click(show_mask)
    
    # --- Layout sections in two columns ---
    # Left column - Common parameters
    left_column = widgets.VBox([
        common_widgets['focal_length_coarse'],
        common_widgets['focal_length_fine'],
        widgets.HBox([common_widgets['M']]),
        common_widgets['two_pi_value'],
        # ROI Quick Set and Center button in one line
        widgets.HBox([roi_selector, center_roi_button], 
                     layout=widgets.Layout(align_items='center', gap='5px')),
        roi_custom,
        # ROI Center X and Y in one line
        widgets.HBox([roi_center_x, roi_center_y], 
                     layout=widgets.Layout(align_items='center', gap='5px')),
    ], layout=widgets.Layout(width='40%', padding='5px'))
    
    # Right column - Mode specific parameters
    optimizer_params = widgets.VBox([
        widgets.HBox([opt_specific_widgets['overlap_ratio'], opt_specific_widgets['center_blend']]),
        widgets.HBox([opt_specific_widgets['interleaving'], opt_specific_widgets['mask_count']]),
        widgets.HBox([opt_specific_widgets['airy_correction'],
                      opt_specific_widgets['dof_correction'],
                      opt_specific_widgets['psf_energy_level']]),
        widgets.HBox([opt_specific_widgets['lr'], opt_specific_widgets['ni']]),
        # Set button
        widgets.HBox([show_mask_button]),
    ], layout={'display': 'flex'})
    
    fresnel_params = widgets.VBox([
        widgets.HTML("<i>No additional parameters required for Fresnel mode</i>")
    ], layout={'display': 'none'})
    
    right_column = widgets.VBox([
        optimizer_params,
        fresnel_params
    ], layout=widgets.Layout(width='58%', padding='5px'))
    
    # --- Dynamic UI toggle ---
    def on_mode_change(change):
        if change.new == 'Optimized Microlens':
            optimizer_params.layout.display = 'flex'
            fresnel_params.layout.display = 'none'
        else:
            optimizer_params.layout.display = 'none'
            fresnel_params.layout.display = 'flex'
    
    # --- Reset button ---
    def reset_to_defaults(_b):
        # Reset common widgets
        for key, default_val in config.COMMON_DEFAULTS.items():
            if key in common_widgets:
                common_widgets[key].value = default_val
        
        # Reset ROI
        roi_selector.value = config.ROI_DEFAULT_SIZE
        roi_custom.value = config.ROI_DEFAULT_SIZE
        
        # Reset ROI position to center
        roi_center_x.value = slm_width // 2
        roi_center_y.value = slm_height // 2
        
        # Reset mode-specific widgets
        if mode_selector.value == 'Optimized Microlens':
            for key, default_val in config.OPTIMIZED_DEFAULTS.items():
                if key in opt_specific_widgets:
                    opt_specific_widgets[key].value = default_val
    
    reset_button = widgets.Button(
        description='Reset to Defaults',
        button_style='warning',
        icon='refresh',
        layout=widgets.Layout(width='150px')
    )
    reset_button.on_click(reset_to_defaults)
    
    # --- Main execution ---
    output_widget = widgets.Output()
    
    # Track if we're currently updating to prevent recursion
    updating = False
    
    def update_parameters(mode):
        """
        Gathers raw values from the UI widgets, then calls the processing function.
        """
        # 1. 从所有GUI控件收集原始值到一个字典
        raw_params = {key: w.value for key, w in common_widgets.items()}
        
        # 手动添加ROI相关的控件值
        raw_params['N'] = roi_custom.value
        raw_params['roi_center_x'] = roi_center_x.value
        raw_params['roi_center_y'] = roi_center_y.value
        
        # 添加SLM的形状信息
        raw_params['shape'] = slm_manager.shape

        # 如果是优化模式，添加特定参数
        if mode == 'Optimized Microlens':
            raw_params.update({key: w.value for key, w in opt_specific_widgets.items()})

        # 2. 调用独立的处理函数来完成计算
        processed_params = process_parameters(raw_params)
        
        return processed_params
            
    
    def run_simulation(_b=None):
        nonlocal updating
        if updating:
            return
        updating = True
        
        try:
            with output_widget:
                clear_output(wait=True)
                mode = mode_selector.value
                print(f"Running mode: {mode}")
                
                # update every field for parameters
                params = update_parameters(mode)
                
                if mode == 'Optimized Microlens':
                    phi, optimizer_obj, info = generate_optimized_pattern(params, plot_live_update)
                    slm_manager.upload(phi)
                    plot_final_results(optimizer_obj,info)
                    
                elif mode == 'Fresnel Microlens':
                    phi, info_dict = generate_fresnel_pattern(params)
                    slm_manager.upload(phi)
                    plot_fresnel_pattern(info_dict)
        finally:
            updating = False
    
    # --- Auto-update functionality ---
    def on_parameter_change(change):
        if auto_update_checkbox.value and not updating:
            run_simulation()
    
    # Attach observers to all parameter widgets
    all_widgets = list(common_widgets.values()) + list(opt_specific_widgets.values()) + [roi_custom, roi_center_x, roi_center_y]
    for widget in all_widgets:
        widget.observe(on_parameter_change, names='value')
    
    # Also trigger on mode change when auto-update is enabled
    def on_mode_change_with_update(change):
        # Handle UI visibility toggle
        if change.new == 'Optimized Microlens':
            optimizer_params.layout.display = 'flex'
            fresnel_params.layout.display = 'none'
        else:
            optimizer_params.layout.display = 'none'
            fresnel_params.layout.display = 'flex'
        
        # Trigger auto-update if enabled
        if auto_update_checkbox.value:
            run_simulation()
    
    mode_selector.observe(on_mode_change_with_update, names='value')
    
    apply_button = widgets.Button(
        description='Generate & Upload',
        button_style='primary',
        icon='cogs',
        layout=widgets.Layout(width='180px')
    )
    apply_button.on_click(run_simulation)
    
    # Initialize ROI position limits
    update_roi_position_limits()
    
    # --- Final layout ---
    ui = widgets.VBox([
        widgets.HBox([mode_selector, auto_update_checkbox], 
                     layout=widgets.Layout(justify_content='center', align_items='center', gap='20px')),
        widgets.HTML("<hr>"),
        widgets.HBox([left_column, right_column]),
        widgets.HTML("<hr>"),
        widgets.HBox([reset_button, apply_button], 
                     layout=widgets.Layout(justify_content='center', gap='20px')),
        output_widget
    ], layout=widgets.Layout(width='100%'))
    
    display(ui)

# --- Run UI ---
build_ui()